In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
q1_2026_power_cut_df = pd.read_csv('/content/power_cut_data_jan_mar_2026.csv')

In [ ]:
q1_2026_power_cut_df['power_cut_hours'] = q1_2026_power_cut_df['power_cut_flag']/60

In [ ]:
q1_2026_power_cut_df['CAIDI'] = q1_2026_power_cut_df['power_cut_hours']/q1_2026_power_cut_df['event_count']

In [ ]:
q1_2026_power_cut_df['CAIDI']

In [ ]:
# Ensure 'timestamp' is a datetime object
q1_2026_power_cut_df['timestamp'] = pd.to_datetime(q1_2026_power_cut_df['timestamp'])

# Extract month as a string for better visualization on the x-axis
q1_2026_power_cut_df['month'] = q1_2026_power_cut_df['timestamp'].dt.strftime('%b %Y')

# Define a custom order for months
month_order = [
    'Jan 2026', 'Feb 2026', 'Mar 2026'
]

# Create a pivot table for the heatmap
heatmap_data = q1_2026_power_cut_df.pivot_table(
    index='site_ID',
    columns='month',
    values='CAIDI'
    #,aggfunc='mean' # Aggregate if multiple entries for a site-month
).reindex(columns=month_order)

import matplotlib.colors as mcolors

# Define the threshold for "usual numbers" (based on data snippet, values are mostly below 2.0)
normal_vmax = 2.0

# Define the explicit outlier value mentioned by the user
explicit_outlier_value = 7.8

# Determine the actual maximum value in the data to ensure the colormap covers all data points.
# This ensures the highest possible value, whether it's from the data or the explicit outlier, sets the scale.
current_max_in_data = heatmap_data.max().max() if not heatmap_data.empty else 0.0
overall_max_for_cmap = max(current_max_in_data, explicit_outlier_value)

# 1. Create the base colormap for the "normal" range (e.g., 'YlOrRd').
base_cmap = plt.colormaps['YlOrRd']

# 2. Define the colors for the custom colormap.
# We want `normal_vmax` to be the breakpoint.
# Let's use two distinct sections for the colormap.
# Section A: 0 to `normal_vmax` (using `base_cmap` colors)
# Section B: `normal_vmax` to `overall_max_for_cmap` (using a distinct outlier color)

# Number of colors to sample for the 'normal' range to create a smooth gradient
num_normal_colors = 100
normal_colors = base_cmap(np.linspace(0, 1, num_normal_colors)).tolist() # Sample colors from base_cmap

outlier_display_color = '#8B0000' # A strong, distinct color for the outlier (Dark Red/Blackish Red)

# Combine normal colors with the outlier color
all_custom_colors = normal_colors + [outlier_display_color]

# Define the boundaries for each color.
# `BoundaryNorm` expects `len(boundaries) == len(colors) + 1`.
# The first `num_normal_colors` boundaries will be from 0 to `normal_vmax`.
# The last boundary will be `overall_max_for_cmap`.

boundaries_for_norm = np.linspace(0, normal_vmax, num_normal_colors).tolist()
# Add the threshold for outlier and the absolute max for the last bin.
# The last bin (with `outlier_display_color`) will be for values from `normal_vmax` up to `overall_max_for_cmap`.
# Adding a small epsilon to the last boundary ensures `overall_max_for_cmap` is included in the last bin.
boundaries_for_norm.append(overall_max_for_cmap + 0.01)

# Create the custom colormap and normalization.
cmap_outlier = mcolors.ListedColormap(all_custom_colors)
norm_outlier = mcolors.BoundaryNorm(boundaries_for_norm, cmap_outlier.N)

# Create the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data,
    cmap=cmap_outlier, # Use the custom colormap
    norm=norm_outlier, # Apply the custom normalization
    annot=True, # Show the CAIDI values on the heatmap
    fmt=".2f", # Format annotation to two decimal places
    linewidths=.5 # Add lines between cells for better separation
)
plt.title('CAIDI (Q1 2026) with Outlier Highlighted')
plt.xlabel('Month')
plt.ylabel('Site ID')
plt.tight_layout()
plt.savefig('reliability_metrics_q1_2026.png')
plt.show()